# Biohub — Cell Tracking · ver 4 chạy trên TRAIN + chấm local scorer

Mục đích: **đo điểm offline** trước khi submit — pipeline ver 4 chạy trên thư mục `train` (có nhãn `.geff`) rồi chấm bằng port trung thành của metric chính thức (adjEJ + 0,1·divJ). Không tốn quota 5 submit/ngày.

**Cách dùng**: File → Import Notebook, Add Input competition, Save & Run All. Cell 2 đã trỏ `DATA_DIR` vào `train`; muốn đổi núm (PERCENTILE, STITCH_*, DIV_*) thì sửa cell 2 rồi Run All lại. So sánh nhanh: ver 1 = 0.198 trên Kaggle, top 1 ≈ 0.97.

Lưu ý: bản notebook NỘP BÀI là `ver4-cell-tracking.ipynb` (đọc `test`, không có cell chấm điểm).


In [ ]:
# ver 4 · cell 1 — IMPORTS
# Dán đè Cell 1 của notebook Kaggle "Biohub - Cell Tracking During Development".
# ============================================================

import itertools
import json
import os
import time

import blosc2
import numpy as np
import pandas as pd
from scipy.ndimage import center_of_mass, label, maximum_filter, uniform_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


In [ ]:
# ver 4 · cell 2 — SIÊU THAM SỐ (mọi núm tune nằm ở đây)
# Dán đè Cell 2 của notebook Kaggle.
# ============================================================
# Mục tiêu ver 4 — NỐI LẠI TRACK ĐỨT (điểm nghẽn số 1 của điểm 0.198):
# chẩn đoán ver 3 trên test: trung vị 2–3 node/track trong khi GT là
# 35 khung/track; ~20% node mở đầu track mới. Ba nguyên nhân + ba cách vá:
#   1. GATE_MIN_UM 5 → 7: p99 bước GT = 6,9µm — gate dưới đó giết cả bước
#      hợp lệ (gate thích ứng = 2,5×median từng vùng, bị kẹp [GATE_MIN, 14]).
#   2. MAX_SKIP_FRAMES 2 → 3: tế bào mờ 3 khung vẫn nối lại được.
#   3. MỚI — STITCHING HẬU KIỂM: chạy hết dataset rồi mới nối mọi track
#      kết thúc ở khung t với track mở đầu ở khung t+gap nếu nằm trong
#      gate + khối lượng tương thích; gap ≥ 2 thì CHÈN NODE NỘI SUY
#      (chỉ tạo cạnh liền khung — cạnh nhảy bị metric bỏ hẳn).
# Phần phát hiện / phân bào giữ nguyên ver 3 (đã kiểm chứng 21/21).
# ============================================================

TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
# Cell 3 đọc DATA_DIR — bản nộp bài giữ = TEST_DIR; bản chạy trên train
# (để chấm bằng local scorer) đổi thành TRAIN_DIR của competition.
TRAIN_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/train'
DATA_DIR = TRAIN_DIR          # bản chạy-train: đọc train (có nhãn .geff)

# Thang vật lý (Z, Y, X) — µm/voxel, theo đề bài
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

# --- Tầng 1 · DETECTION (giữ nguyên ver 3) ---
DS_Z, DS_Y, DS_X = 2, 4, 4       # downsample bất đối xứng: z giữ kỹ hơn
SMOOTH_SIZE = 3                   # uniform_filter, trong không gian downsample
PERCENTILE = 90.0                 # bắt thêm nhân mờ (recall > precision)
MIN_NVOXELS = 4                   # bỏ component li ti (nhiễu)
MAX_NVOXELS = 3000                # blob gộp vẫn nhận, tách ở tầng 1b
CONN26 = True                     # liên kết 26-ô: chống tách nhân giữa các lát z

# --- Tầng 1b · TÁCH BLOB GỘP (giữ nguyên ver 3) ---
SPLIT_MIN_NVOXELS = 90            # component ds lớn hơn mới xét tách
PEAK_SIZE = (3, 5, 5)             # maximum_filter tìm đỉnh cục bộ (z,y,x ds)
MIN_PEAK_DIST_DS = 4.0            # 2 đỉnh cách ≥ 4 voxel ds
PEAK_MIN_BRIGHT = 1.15            # đỉnh sáng hơn ngưỡng ít nhất 15%

# --- Tầng 2 · LINKING (ver 4: GATE_MIN 5 → 7) ---
BASE_GATE_UM = 9.0
GATE_MEDIAN_MULT = 2.5
GATE_MIN_UM, GATE_MAX_UM = 7.0, 14.0   # p99 bước GT 6,9µm — min 5 giết bước hợp lệ
VEL_SMOOTH = 0.5                  # vận tốc EMA: v ← 0.5·mới + 0.5·cũ
BRIGHT_WEIGHT = 0.0               # phạt chênh độ sáng trong cost (tắt)

# --- Frame-skip + nội suy (ver 4: MAX_SKIP 2 → 3) ---
ALLOW_FRAME_SKIP = True
SKIP_GATE_UM = 12.0
MAX_SKIP_FRAMES = 3
INTERPOLATE_MISSED_FRAMES = True  # GT 100% cạnh liền khung → nội suy bắt buộc
TIME_LIMIT_HOURS = 11.0

# --- MỚI ver 4 · STITCHING HẬU KIỂM (chạy trong cell 3) ---
STITCH_ENABLED = True
STITCH_GAP_MAX = 5                    # nối lại track đứt cách ≤ 5 khung
STITCH_GATE_UM = 10.0                 # gate tại gap=1: bắt bước 7–10µm
STITCH_GATE_PER_GAP = 2.0             # gate(g) = 10 + 2·(g−1) µm (g=5 → 18)
STITCH_MAX_LOGMASS = 1.1              # |ln(m_start/m_end)| ≤ 1,1 (~3×)
STITCH_COLLISION_UM = 0.0             # 0 = tắt: hành lang nối blob-break đi đúng
                                      # qua node CoM của track bạn đồng hành

# --- Tầng 3 · PHÂN BÀO THEO PROFILE ĐỘ SÁNG (giữ nguyên ver 3) ---
DIVISION_ENABLED = True
DIV_PARENT_GATE_UM = 12.0         # cửa sổ tìm kiếm (base rate 24:1)
DIV_SIBLING_GATE_UM = 14.5       # p99 sister sep 13,9, max 14,65
DIV_MIN_CHILD_FRAC = 0.15
DIV_BRIGHTNESS_CHECK = True
DIV_BRIGHTNESS_RATIO = (0.55, 1.8)
DIV_CONFIRM_FRAMES = 3
DIV_SEP_GROWTH = 1.15
MASS_HISTORY = 12
MASS_SKIP_LAST = 2
DIV_MOM_MAX_RISE = 1.7
DIV_MOM_MIN_FRAC = 0.5
DIV_MOM_BRIGHT_BONUS = 1.05
MASS_BASE_MIN_FRAMES = 4

# --- Cấu hình LOCAL SCORER (các cell cuối notebook dùng) ---
SUBMISSION_CSV = 'submission.csv'   # cell 4 vừa ghi
MAX_DATASETS = 0                    # 0 = tất cả dataset train; >0 = chạy nhanh
MAX_DISTANCE = 7.0                   # ngưỡng ghép node (µm) — đúng đề bài
ADJUSTMENT_ALPHA = 0.1               # hệ số phạt node thừa (metric chính thức)
SCORE_DIVISION_WEIGHT = 0.1         # trọng số division trong điểm tổng

# --- Soát bằng mắt (vẽ 1 khung đầu tiên) ---
RUN_PREVIEW = False

# --- Chẩn đoán ---
DIAGNOSE = True

NODE_ID = itertools.count(1)      # node_id duy nhất toàn cục
BIG = 1e9

# Tín hiệu cho cell 3: cell 2 ver-4 đã chạy trong notebook này → cell 3 dùng
# đúng núm ở trên (cell 3 có sẵn bộ mặc định y hệt, phòng khi bị dán đè đơn lẻ
# vào notebook chưa từng chạy cell 1/cell 2 ver-4 — hotfix 14/09).
PIPELINE_CONFIG_VERSION = 4


In [ ]:
# ver 4 · cell 3 — PIPELINE: ĐỌC ZARR → DETECTION (+TÁCH BLOB) → TRACKING
#                          → PHÂN BÀO THEO PROFILE ĐỘ SÁNG → STITCHING HẬU KIỂM
# Dán đè Cell 3 của notebook Kaggle (toàn bộ thuật toán nằm ở đây).
# ============================================================
# Thay đổi so với ver 3 (nhắm thẳng điểm nghẽn số 1 của 0.198 — track đứt):
#   * GATE_MIN_UM 5 → 7 (p99 bước GT 6,9µm — xem cell 2).
#   * MAX_SKIP_FRAMES 2 → 3.
#   * MỚI — STITCHING HẬU KIỂM: sau khi chạy hết dataset, mọi track kết
#     thúc ở khung t (node không có cạnh ra) được nối với track mở đầu ở
#     khung t+gap (node không có cạnh vào) nếu nằm trong gate + khối lượng
#     tương thích; gap ≥ 2 thì chèn node nội suy → chỉ cạnh liền khung.
#     Ba tình huống được cứu: (a) tế bào mờ > MAX_SKIP_FRAMES khung;
#     (b) blob gộp > MAX_SKIP_FRAMES khung làm track bạn đồng hành chết;
#     (c) tế bào quẹo khi mờ → dự đoán pos+vel·gap trượt khỏi SKIP_GATE.
# Phát hiện / phân bào giữ nguyên ver 3 (mass stability + 5 cửa).
# ============================================================
# Hotfix 13/09: cấu trúc 26-liên kết tạo TRỰC TIẾP trong detect_nodes —
# Cell 3 tự chứa, dán thiếu dòng đầu không còn gây NameError.
# Hotfix 14/09: TỰ CHỨA HOÀN TOÀN — cell 3 tự import (maximum_filter & co.)
# và tự kèm bộ cấu hình mặc định ver 4 → dán đè ĐƠN LẼ cell 3 vào notebook
# đang giữ cell 1/cell 2 BẢN CŨ vẫn chạy đúng (lỗi thực tế trên Kaggle:
# cell 1 cũ thiếu `maximum_filter` → NameError ngay khung đầu tiên).
# Cell 2 ver-4 đã chạy thì núm tune ở đó vẫn thắng (xem khối 0b bên dưới).
# ============================================================

# ---------- 0) IMPORT TỰ CHỮA ----------
# Import lại là vô hại (module đã load thì Python lấy từ cache) — cell 1
# ver-4 có dòng import trùng cũng không đổi hành vi gì.
import itertools
import json
import os
import time

import numpy as np
import pandas as pd
from scipy.ndimage import center_of_mass, label, maximum_filter, uniform_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

try:
    import blosc2
except ImportError as _e:
    raise ImportError(
        'Chưa có blosc2 — hãy chạy Cell 1 (pip install) trước rồi chạy lại '
        'cell này, hoặc gõ  !pip install -q blosc2  trong 1 cell riêng.'
    ) from _e

# ---------- 0b) CẤU HÌNH MẶC ĐỊNH ver 4 (phòng khi chỉ dán đè cell 3) ----------
# CHỈ áp khi cell 2 ver-4 CHƯA chạy trong notebook này (chưa có
# PIPELINE_CONFIG_VERSION) — cell 2 ver-4 đã chạy thì giữ nguyên núm của nó
# (tune ở cell 2). DATA_DIR đã được ai đó đặt sẵn (bản chạy-train, test cục
# bộ) cũng được tôn trọng. Bộ giá trị bên dưới PHẢI GIỮ ĐỒNG BỘ với cell 2.
if globals().get('PIPELINE_CONFIG_VERSION', 0) < 4:
    if 'DATA_DIR' not in globals():
        TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
        # bản nộp bài đọc test; muốn chấm bằng local scorer thì trỏ sang train
        DATA_DIR = TEST_DIR

    # Thang vật lý (Z, Y, X) — µm/voxel, theo đề bài
    SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

    # --- Tầng 1 · DETECTION ---
    DS_Z, DS_Y, DS_X = 2, 4, 4       # downsample bất đối xứng: z giữ kỹ hơn
    SMOOTH_SIZE = 3                  # uniform_filter, trong không gian downsample
    PERCENTILE = 90.0                # bắt thêm nhân mờ (recall > precision)
    MIN_NVOXELS = 4                  # bỏ component li ti (nhiễu)
    MAX_NVOXELS = 3000               # blob gộp vẫn nhận, tách ở tầng 1b
    CONN26 = True                    # liên kết 26-ô: chống tách nhân giữa các lát z

    # --- Tầng 1b · TÁCH BLOB GỘP ---
    SPLIT_MIN_NVOXELS = 90           # component ds lớn hơn mới xét tách
    PEAK_SIZE = (3, 5, 5)            # maximum_filter tìm đỉnh cục bộ (z,y,x ds)
    MIN_PEAK_DIST_DS = 4.0           # 2 đỉnh cách ≥ 4 voxel ds
    PEAK_MIN_BRIGHT = 1.15           # đỉnh sáng hơn ngưỡng ít nhất 15%

    # --- Tầng 2 · LINKING (ver 4: GATE_MIN 5 → 7) ---
    BASE_GATE_UM = 9.0
    GATE_MEDIAN_MULT = 2.5
    GATE_MIN_UM, GATE_MAX_UM = 7.0, 14.0   # p99 bước GT 6,9µm — min 5 giết bước hợp lệ
    VEL_SMOOTH = 0.5                 # vận tốc EMA: v ← 0.5·mới + 0.5·cũ
    BRIGHT_WEIGHT = 0.0              # phạt chênh độ sáng trong cost (tắt)

    # --- Frame-skip + nội suy (ver 4: MAX_SKIP 2 → 3) ---
    ALLOW_FRAME_SKIP = True
    SKIP_GATE_UM = 12.0
    MAX_SKIP_FRAMES = 3
    INTERPOLATE_MISSED_FRAMES = True  # GT 100% cạnh liền khung → nội suy bắt buộc
    TIME_LIMIT_HOURS = 11.0

    # --- ver 4 · STITCHING HẬU KIỂM (chạy trong cell 3) ---
    STITCH_ENABLED = True
    STITCH_GAP_MAX = 5               # nối lại track đứt cách ≤ 5 khung
    STITCH_GATE_UM = 10.0            # gate tại gap=1: bắt bước 7–10µm
    STITCH_GATE_PER_GAP = 2.0        # gate(g) = 10 + 2·(g−1) µm (g=5 → 18)
    STITCH_MAX_LOGMASS = 1.1         # |ln(m_start/m_end)| ≤ 1,1 (~3×)
    STITCH_COLLISION_UM = 0.0        # 0 = tắt: hành lang nối blob-break đi đúng
                                     # qua node CoM của track bạn đồng hành

    # --- Tầng 3 · PHÂN BÀO THEO PROFILE ĐỘ SÁNG ---
    DIVISION_ENABLED = True
    DIV_PARENT_GATE_UM = 12.0        # cửa sổ tìm kiếm (base rate 24:1)
    DIV_SIBLING_GATE_UM = 14.5       # p99 sister sep 13,9, max 14,65
    DIV_MIN_CHILD_FRAC = 0.15
    DIV_BRIGHTNESS_CHECK = True
    DIV_BRIGHTNESS_RATIO = (0.55, 1.8)
    DIV_CONFIRM_FRAMES = 3
    DIV_SEP_GROWTH = 1.15
    MASS_HISTORY = 12
    MASS_SKIP_LAST = 2
    DIV_MOM_MAX_RISE = 1.7
    DIV_MOM_MIN_FRAC = 0.5
    DIV_MOM_BRIGHT_BONUS = 1.05
    MASS_BASE_MIN_FRAMES = 4

    # --- Soát bằng mắt (vẽ 1 khung đầu tiên) — tôn trọng ai đã tắt trước đó ---
    RUN_PREVIEW = globals().get('RUN_PREVIEW', True)

    # --- Chẩn đoán ---
    DIAGNOSE = True

    NODE_ID = itertools.count(1)     # node_id duy nhất toàn cục
    BIG = 1e9

    PIPELINE_CONFIG_VERSION = 4      # đánh dấu đã áp cấu hình ver-4

T_START = time.time()


def _over_time_budget():
    return (time.time() - T_START) > TIME_LIMIT_HOURS * 3600.0


# ---------- 1) ĐỌC DỮ LIỆU (giữ nguyên ver 2) ----------
def read_zarr_meta(zarr_path):
    """Đọc shape/dtype/chunk grid từ zarr.json của mảng (như notebook gốc)."""
    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        meta = json.load(f)
    shape = tuple(int(v) for v in meta['shape'])            # (T, Z, Y, X)
    dtype = np.dtype(meta['data_type'])
    chunk = meta.get('chunk_grid', {}).get('configuration', {}).get('chunk_shape')
    if not chunk or len(chunk) != 4:
        chunk = [1, shape[1], shape[2], shape[3]]
    return shape, dtype, [int(c) for c in chunk]


def load_volume(zarr_path, t, shape, dtype, chunk):
    """Đọc khung t → (Z, Y, X). Chunk 0/0/0 là đường nhanh như notebook gốc;
    nếu khung được chia nhiều chunk thì lắp ghép đủ các chunk có mặt."""
    T, Z, Y, X = shape
    _, cZ, cY, cX = chunk
    croot = os.path.join(zarr_path, '0', 'c', str(t))
    if not os.path.isdir(croot):
        raise FileNotFoundError(f'không tìm thấy khung t={t}: {croot}')

    def read_chunk(cz, cy, cx):
        p = os.path.join(croot, str(cz), str(cy), str(cx))
        with open(p, 'rb') as f:
            return np.frombuffer(blosc2.decompress(f.read()), dtype=dtype)

    # Đường nhanh: metadata nói 1 chunk chứa cả khung
    if (cZ, cY, cX) == (Z, Y, X):
        try:
            arr = read_chunk(0, 0, 0)
            if arr.size == Z * Y * X:
                return arr.reshape(Z, Y, X)
        except Exception:
            pass  # rơi xuống đường đa chunk bên dưới

    vol = np.zeros((Z, Y, X), dtype=dtype)
    placed = 0
    for cz in range((Z + cZ - 1) // cZ):
        for cy in range((Y + cY - 1) // cY):
            for cx in range((X + cX - 1) // cX):
                p = os.path.join(croot, str(cz), str(cy), str(cx))
                if not os.path.isfile(p):
                    continue
                try:
                    arr = read_chunk(cz, cy, cx)
                except Exception:
                    continue
                vz = min(cZ, Z - cz * cZ)
                vy = min(cY, Y - cy * cY)
                vx = min(cX, X - cx * cX)
                if vz <= 0 or vy <= 0 or vx <= 0:
                    continue
                if arr.size >= cZ * cY * cX:      # chunk đầy (zarr v3 pad theo fill)
                    block = arr[:cZ * cY * cX].reshape(cZ, cY, cX)[:vz, :vy, :vx]
                elif arr.size >= vz * vy * vx:     # biến thể chunk gọn ở biên
                    block = arr[:vz * vy * vx].reshape(vz, vy, vx)
                else:
                    continue
                vol[cz * cZ:cz * cZ + vz, cy * cY:cy * cY + vy, cx * cX:cx * cX + vx] = block
                placed += 1
    if placed == 0:
        raise FileNotFoundError(f'khung t={t}: không đọc được chunk nào từ {croot}')
    return vol


# ---------- 2) DETECTION + TÁCH BLOB GỘP (giữ nguyên ver 2) ----------
def _com(coords, w):
    """Center-of-mass theo cường độ của tập voxel (z,y,x ds, float)."""
    s = w.sum()
    if s <= 0:
        return coords.mean(axis=0)
    return (coords * w[:, None]).sum(axis=0) / s


def detect_nodes(vol):
    """Trả về list dict: z, y, x (voxel gốc, int — đúng format đề bài) và
    zf, yf, xf (float — cho tracking), mass (tổng cường độ), nvox."""
    ds = vol[::DS_Z, ::DS_Y, ::DS_X].astype(np.float32)
    if ds.size == 0:
        return []
    smoothed = uniform_filter(ds, size=SMOOTH_SIZE)
    thr = np.percentile(smoothed, PERCENTILE)
    mask = smoothed > thr
    labeled, n = label(mask, structure=np.ones((3, 3, 3), dtype=bool) if CONN26 else None)
    if n == 0:
        return []

    # đỉnh cục bộ (tính 1 lần cả khung): voxel = max trong lân cận PEAK_SIZE
    # và sáng hơn ngưỡng ít nhất PEAK_MIN_BRIGHT (chống đỉnh nhiễu)
    mf = maximum_filter(smoothed, size=PEAK_SIZE)
    is_peak = (smoothed == mf) & mask & (smoothed >= thr * PEAK_MIN_BRIGHT)

    lab_flat = labeled.ravel()
    counts = np.bincount(lab_flat, minlength=n + 1)
    wsums = np.bincount(lab_flat, weights=smoothed.ravel(), minlength=n + 1).astype(np.float64)
    coms = center_of_mass(smoothed, labeled, index=range(1, n + 1))

    # trọng số khoảng cách trong không gian ds: z gấp đôi (bước z dài gấp đôi)
    ZW = np.array([2.0, 1.0, 1.0])

    nodes = []
    for i in range(1, n + 1):
        nv = int(counts[i])
        if nv < MIN_NVOXELS or nv > MAX_NVOXELS:
            continue

        split_done = False
        if nv > SPLIT_MIN_NVOXELS:
            comp = np.argwhere(labeled == i)                # (N, 3) z,y,x
            pk_mask = is_peak[comp[:, 0], comp[:, 1], comp[:, 2]]
            peaks = comp[pk_mask]
            if len(peaks) >= 2:
                # giữ các đỉnh sáng nhất, bỏ đỉnh quá gần một đỉnh đã giữ
                vals = smoothed[peaks[:, 0], peaks[:, 1], peaks[:, 2]]
                order = np.argsort(-vals)
                kept = []
                for idx in order:
                    p = peaks[idx].astype(np.float64)
                    too_close = False
                    for q in kept:
                        if np.linalg.norm((p - q) * ZW) < MIN_PEAK_DIST_DS:
                            too_close = True
                            break
                    if not too_close:
                        kept.append(p)
                if len(kept) >= 2:
                    # gán mỗi voxel component → đỉnh gần nhất; CoM riêng từng vùng
                    P = np.array(kept)
                    D = cdist(comp.astype(np.float64) * ZW, P * ZW)
                    assign = D.argmin(axis=1)
                    w_all = smoothed[comp[:, 0], comp[:, 1], comp[:, 2]].astype(np.float64)
                    for k in range(len(kept)):
                        sel = assign == k
                        sub = comp[sel]
                        w = w_all[sel]
                        if sub.shape[0] < MIN_NVOXELS:
                            continue
                        cz, cy, cx = _com(sub.astype(np.float64), w)
                        nodes.append({
                            'z': int(round(cz * DS_Z)), 'y': int(round(cy * DS_Y)),
                            'x': int(round(cx * DS_X)),
                            'zf': cz * DS_Z, 'yf': cy * DS_Y, 'xf': cx * DS_X,
                            'mass': float(w.sum()), 'nvox': int(sub.shape[0]),
                        })
                    split_done = True
        if split_done:
            continue

        cz, cy, cx = coms[i - 1]
        nodes.append({
            'z': int(round(cz * DS_Z)), 'y': int(round(cy * DS_Y)), 'x': int(round(cx * DS_X)),
            'zf': cz * DS_Z, 'yf': cy * DS_Y, 'xf': cx * DS_X,
            'mass': float(wsums[i]), 'nvox': nv,
        })
    return nodes


# ---------- 3) TRACKER (ver 3: profile độ sáng cho phân bào) ----------
def _mass_baseline(hist):
    """Baseline khối lượng của track: PERCENTILE 25 của MASS_HISTORY giá trị
    gần nhất, BỎ MASS_SKIP_LAST khung cuối (đúng khoảng mẹ sáng lên trước
    khi chia). Dùng p25 thay vì median: blob gộp merge-split (≈2×) tồn tại
    nửa cửa sổ vẫn không kéo được baseline lên — median thì được (bug đã
    bắt gặp khi kiểm chứng trên mô phỏng TS), còn p25 vẫn kháng được 1–2
    khung mờ nháp. Trả về None nếu chưa đủ dữ liệu."""
    if len(hist) < MASS_BASE_MIN_FRAMES + MASS_SKIP_LAST:
        return None
    kept = hist[-(MASS_HISTORY + MASS_SKIP_LAST):-MASS_SKIP_LAST]
    if not kept:
        return None
    return float(np.percentile(kept, 25))


class Tracker:
    """Hungarian + motion model + tách blob + PHÂN BÀO THEO PROFILE ĐỘ SÁNG.

    Mỗi khung gồm 5 bước:
      (a) Hungarian giữa node khung trước (đặt tại vị trí DỰ ĐOÁN) và node
          khung hiện tại, gate thích ứng
      (b) phân bào ỨNG VIÊN — 5 CỬA:
          1. parent gate 12µm (cửa sổ tìm kiếm — base rate 24:1 nên khoảng
             cách không còn là tín hiệu)
          2. sibling gate 14,5µm (p99 thật 13,9 — bắt cả cặp chị em xa)
          3. bảo toàn độ sáng (con1+con2 ≈ mẹ) + con thứ 2 ≥ 15% mẹ
          4. MỚI — ỔN ĐỊNH KHỐI LƯỢNG MẸ: khối lượng blob mẹ tại khung tách
             ≤ 1,7× baseline riêng của nó (blob gộp 2 tế bào ≈ 2×; mẹ thật
             chỉ sáng lên nhẹ) và ≥ 0,5× (mẹ không phai đột ngột)
          5. MỚI — Ưu tiên appearance: ứng viên có mẹ sáng dần ≥ 5% xếp trước
          Cạnh divergence vẫn HOÃN chờ bước (e)
      (c) frame-skip: nối lại track mất ≤ 2 khung + nội suy node giữa
      (d) dọn dẹp
      (e) xác nhận động học: 2 con sống ≥ DIV_CONFIRM_FRAMES khung và
          khoảng cách tăng ≥ 15% — cạnh + division chỉ ghi khi vượt hết
    """

    def __init__(self):
        self.active = {}        # nid → dict(pos, vel, mass, mass_hist)
        self.pending = {}       # nid → dict(pos, vel, mass, mass_hist, missed)
        self.recent_steps = []  # bước đi (µm) của các match gần đây
        self.div_pending = []   # ứng viên phân bào chờ xác nhận
        self.n_div_confirmed = 0
        self.n_div_rejected = 0
        # ver 3: đếm lý do từ chối để chẩn đoán
        self.n_rej_mass = 0        # rớt ổn định khối lượng mẹ (bước b — MỚI)
        self.n_rej_lost = 0        # một con biến mất (bước e)
        self.n_rej_dyn = 0         # không tách ra đủ (bước e)

    def _gate(self):
        if len(self.recent_steps) < 5:
            return BASE_GATE_UM
        med = float(np.median(self.recent_steps))
        return float(np.clip(GATE_MEDIAN_MULT * med, GATE_MIN_UM, GATE_MAX_UM))

    @staticmethod
    def _push_hist(hist, m):
        h = list(hist)
        h.append(float(m))
        if len(h) > MASS_HISTORY + MASS_SKIP_LAST + 2:
            del h[:len(h) - (MASS_HISTORY + MASS_SKIP_LAST + 2)]
        return h

    def step(self, dets, t):
        """Xử lý khung t. Trả về (nodes, edges, divisions, interp_nodes)."""
        nodes = [(next(NODE_ID), d) for d in dets]
        ids = [nid for nid, _ in nodes]
        pos = np.array(
            [[d['zf'], d['yf'], d['xf']] for _, d in nodes], dtype=np.float64,
        ).reshape(-1, 3) * SCALE
        mass = [float(d['mass']) for _, d in nodes]

        edges, divisions, interp_nodes = [], [], []
        matched_curr = {}   # curr_idx → prev nid
        succ = {}           # prev nid → nid khung này (nối mạch cho bước (e))

        # ---- (a) matching chính: Hungarian trên vị trí DỰ ĐOÁN ----
        prev_ids = list(self.active.keys())
        if prev_ids and ids:
            pred = np.array(
                [self.active[p]['pos'] + self.active[p]['vel'] for p in prev_ids],
            )
            D = np.linalg.norm(pred[:, None, :] - pos[None, :, :], axis=2)
            gate = self._gate()
            cost = D
            if BRIGHT_WEIGHT > 0:
                mp = np.maximum(
                    np.array([self.active[p]['mass'] for p in prev_ids])[:, None], 1e-9)
                mc = np.maximum(np.array(mass)[None, :], 1e-9)
                cost = D * (1.0 + BRIGHT_WEIGHT * np.abs(np.log(mc / mp)))
            cost = np.where(D <= gate, cost, BIG)
            ri, ci = linear_sum_assignment(cost)
            for r, c in zip(ri, ci):
                if D[r, c] > gate:
                    continue
                p = prev_ids[r]
                matched_curr[c] = p
                succ[p] = ids[c]
                edges.append((p, ids[c]))
                self.recent_steps.append(
                    float(np.linalg.norm(pos[c] - self.active[p]['pos'])))
            if len(self.recent_steps) > 200:
                del self.recent_steps[:100]

        new_active = {}
        for c, p in matched_curr.items():
            a = self.active[p]
            disp = pos[c] - a['pos']
            vel = VEL_SMOOTH * disp + (1.0 - VEL_SMOOTH) * a['vel']
            # ver 3: lịch sử khối lượng đi THEO TRACK (chuyển từ nid cũ sang nid mới)
            new_active[ids[c]] = {
                'pos': pos[c], 'vel': vel, 'mass': mass[c],
                'mass_hist': self._push_hist(a['mass_hist'], mass[c]),
            }

        # ---- (b) phân bào ỨNG VIÊN (5 cửa — cạnh hoãn chờ bước (e)) ----
        unmatched = [j for j in range(len(ids)) if j not in matched_curr]
        if DIVISION_ENABLED and matched_curr:
            cands = []
            A_vel_cache = {p: self.active[p]['vel'] for p in matched_curr.values()}
            for c, p in matched_curr.items():
                A = self.active[p]
                P, mP, hist = A['pos'], A['mass'], A['mass_hist']
                C1, mC1 = pos[c], mass[c]
                base = _mass_baseline(hist)          # None nếu track quá non
                for j in unmatched:
                    B2, mB2 = pos[j], mass[j]
                    dP = float(np.linalg.norm(P - B2))
                    if dP > DIV_PARENT_GATE_UM:
                        continue
                    dS = float(np.linalg.norm(C1 - B2))
                    if dS > DIV_SIBLING_GATE_UM:
                        continue
                    if mP > 0:
                        if mB2 < DIV_MIN_CHILD_FRAC * mP:
                            continue
                        if DIV_BRIGHTNESS_CHECK:
                            ratio = (mC1 + mB2) / mP
                            if not (DIV_BRIGHTNESS_RATIO[0] <= ratio <= DIV_BRIGHTNESS_RATIO[1]):
                                continue
                    # --- MỚI ver 3: cửa 4 — ổn định khối lượng mẹ ---
                    rise = (mP / base) if (base and base > 0) else None
                    if rise is not None:
                        if rise > DIV_MOM_MAX_RISE:
                            # blob "mẹ" gộp ~2 tế bào → merge-split giả
                            self.n_rej_mass += 1
                            continue
                        if rise < DIV_MOM_MIN_FRAC:
                            # mẹ phai đột ngột → detection không ổn định
                            self.n_rej_mass += 1
                            continue
                    # --- MỚI ver 3: cửa 5 (soft) — ưu tiên mẹ sáng dần ---
                    bright = (rise is not None and rise >= DIV_MOM_BRIGHT_BONUS)
                    cands.append((0 if bright else 1, dP, p, c, j, rise))
            # sort: (mẹ sáng dần trước, rồi tới khoảng cách)
            cands.sort()
            used_p, used_j = set(), set()
            for _prio, dP, p, c, j, rise in cands:
                if p in used_p or j in used_j:
                    continue
                used_p.add(p)
                used_j.add(j)
                # con thứ 2 khởi động như track mới; cạnh p→ids[j] HOÃN,
                # chỉ ghi nếu được xác nhận ở bước (e) của các khung sau
                # QUAN TRỌNG: con KẾ THỪA vận tốc mẹ — không phải vectơ
                # mẹ→con! (vectơ đó làm dự đoán khung sau vọt xa vị trí thật
                # → con không match được → ứng viên chết "lost" và bị đề
                # xuất lại mỗi khung — vòng lặp đã bắt gặp khi kiểm chứng)
                new_active[ids[j]] = {
                    'pos': pos[j], 'vel': np.array(A_vel_cache[p]), 'mass': mass[j],
                    'mass_hist': [mass[j]],
                }
                self.div_pending.append({
                    't': t, 'mother': p, 'mother_mass': self.active[p]['mass'],
                    'mother_rise': rise,
                    'c1': ids[c], 'c2': ids[j], 'c2_start': ids[j],
                    'd0': float(np.linalg.norm(pos[c] - pos[j])), 'age': 0,
                })
            unmatched = [j for j in unmatched if j not in used_j]

        # ---- (c) frame-skip: nối lại track mất ≤ MAX_SKIP_FRAMES khung ----
        if ALLOW_FRAME_SKIP and self.pending and unmatched:
            pend_ids = list(self.pending.keys())
            pred = np.array([
                self.pending[q]['pos'] + self.pending[q]['vel'] * (self.pending[q]['missed'] + 1)
                for q in pend_ids
            ])
            C = pos[np.array(unmatched, dtype=int)]
            D = np.linalg.norm(pred[:, None, :] - C[None, :, :], axis=2)
            cost = np.where(D <= SKIP_GATE_UM, D, BIG)
            ri, ci = linear_sum_assignment(cost)
            for r, c in zip(ri, ci):
                if D[r, c] > SKIP_GATE_UM:
                    continue
                q, j = pend_ids[r], unmatched[c]
                Q = self.pending[q]
                gap = Q['missed'] + 1
                if INTERPOLATE_MISSED_FRAMES and gap >= 2:
                    # Chèn node nội suy tại các khung bị mất rồi nối các cạnh
                    # LIỀN KHUNG — cạnh nhảy t→t+2 bị metric bỏ hẳn.
                    chain = q
                    for k in range(1, gap):
                        pm = Q['pos'] + (pos[j] - Q['pos']) * (k / gap)  # µm
                        vf = pm / SCALE
                        mid = next(NODE_ID)
                        interp_nodes.append((mid, t - gap + k, {
                            'z': int(round(vf[0])), 'y': int(round(vf[1])), 'x': int(round(vf[2])),
                            'zf': float(vf[0]), 'yf': float(vf[1]), 'xf': float(vf[2]),
                            'mass': (Q['mass'] + mass[j]) / 2.0, 'nvox': 0,
                        }))
                        edges.append((chain, mid))
                        chain = mid
                    edges.append((chain, ids[j]))
                new_active[ids[j]] = {
                    'pos': pos[j],
                    'vel': (pos[j] - Q['pos']) / gap,
                    'mass': mass[j],
                    'mass_hist': self._push_hist(Q['mass_hist'], mass[j]),
                }
                succ[q] = ids[j]
                self.pending.pop(q, None)
            unmatched = [j for j in unmatched if ids[j] not in new_active]

        # ---- (d) dọn dẹp ----
        for q in list(self.pending.keys()):
            self.pending[q]['missed'] += 1
            if self.pending[q]['missed'] > MAX_SKIP_FRAMES:
                del self.pending[q]

        matched_prev = set(matched_curr.values())
        for p in prev_ids:
            if p in matched_prev:
                continue
            a = self.active[p]
            self.pending[p] = {'pos': a['pos'], 'vel': a['vel'],
                               'mass': a['mass'], 'mass_hist': a['mass_hist'],
                               'missed': 1}

        for j in unmatched:
            if ids[j] not in new_active:
                new_active[ids[j]] = {
                    'pos': pos[j], 'vel': np.zeros(3), 'mass': mass[j],
                    'mass_hist': [mass[j]],
                }

        # ---- (e) xác nhận phân bào: đủ tuổi + khoảng cách 2 con tăng ----
        still_pending = []
        for cand in self.div_pending:
            if cand['age'] == 0:
                # candidate vừa tạo ở bước (b) của CHÍNH step này — c1/c2 là
                # node của khung hiện tại, chưa bao giờ là "prev" nên chưa
                # có trong succ; chỉ tăng tuổi và chờ step sau
                cand['age'] = 1
                still_pending.append(cand)
                continue
            c1 = succ.get(cand['c1'])
            c2 = succ.get(cand['c2'])
            if c1 is None or c2 is None:
                self.n_div_rejected += 1          # một "con" biến mất → bỏ
                self.n_rej_lost += 1
                continue
            cand['c1'], cand['c2'] = c1, c2
            cand['age'] += 1
            if cand['age'] > DIV_CONFIRM_FRAMES:
                a1, a2 = new_active.get(c1), new_active.get(c2)
                if a1 is not None and a2 is not None:
                    d_now = float(np.linalg.norm(a1['pos'] - a2['pos']))
                    if d_now >= DIV_SEP_GROWTH * cand['d0']:
                        # XÁC NHẬN: ghi cạnh hoãn (mẹ t-1 → con thứ 2 t) —
                        # vẫn là cạnh liền khung vì mẹ/con1/con2 sinh cùng lượt
                        edges.append((cand['mother'], cand['c2_start']))
                        divisions.append((cand['mother'], cand['c2_start']))
                        self.n_div_confirmed += 1
                        continue
                self.n_div_rejected += 1          # không tách ra → merge-split giả
                self.n_rej_dyn += 1
                continue
            still_pending.append(cand)
        self.div_pending = still_pending

        self.active = new_active
        return nodes, edges, divisions, interp_nodes


# ---------- 4) CHẠY TOÀN BỘ TEST SET + CHẨN ĐOÁN ----------
def diagnose(rows):
    """Thống kê "sức khoẻ" submission — phát hiện sớm lỗi cấu trúc.
    Trả về chuỗi log để in cùng kết quả dataset."""
    nd = [r for r in rows if r['row_type'] == 'node']
    ed = [r for r in rows if r['row_type'] == 'edge']
    if not nd:
        return 'không có node'
    node_ids = {r['node_id'] for r in nd}
    has_in = {r['target_id'] for r in ed if r['target_id'] in node_ids}
    forks = {}
    for r in ed:
        if r['source_id'] in node_ids:
            forks[r['source_id']] = forks.get(r['source_id'], 0) + 1
    n_div = sum(1 for v in forks.values() if v >= 2)
    # track = thành phần liên thông yếu
    parent = {i: i for i in node_ids}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for r in ed:
        s, d = r['source_id'], r['target_id']
        if s in parent and d in parent:
            parent[find(s)] = find(d)
    from collections import Counter
    sizes = Counter(find(i) for i in node_ids)
    med_track = float(np.median(list(sizes.values()))) if sizes else 0.0
    n_no_in = sum(1 for r in nd if r['node_id'] not in has_in)
    t_lo = min(r['t'] for r in nd)
    t_hi = max(r['t'] for r in nd)
    frames = t_hi - t_lo + 1
    return (f"{len(nd)} node · {len(ed)} cạnh · {n_div} phân bào · "
            f"{n_no_in}/{len(nd)} node không cạnh vào ({100.0 * n_no_in / len(nd):.0f}%) · "
            f"{len(sizes)} track · trung vị {med_track:.0f} node/track · "
            f"{len(nd) / max(1, frames):.1f} node/khung · "
            f"phân bào xác nhận/từ chối: {TRK.n_div_confirmed}/{TRK.n_div_rejected} "
            f"(rớt mass: {TRK.n_rej_mass} · rớt động học: {TRK.n_rej_dyn} · "
            f"mất con: {TRK.n_rej_lost})")


# ---------- 4.5) STITCHING HẬU KIỂM (mới ver 4) ----------
def stitch_tracks(rows, meta):
    """Nối lại track đứt SAU khi chạy hết dataset — xem cell 2 (ver 4).

    end = node không có cạnh ra; start = node không có cạnh vào. Với mỗi
    gap = 1..STITCH_GAP_MAX (xét từ nhỏ đến lớn): mọi cặp (end ở khung t,
    start ở khung t+gap) trong gate STITCH_GATE_UM + STITCH_GATE_PER_GAP·
    (gap−1) và |ln khối lượng| ≤ STITCH_MAX_LOGMASS → Hungarian toàn cục
    trên nhóm (một end ghép một start) → gap ≥ 2 chèn node nội suy tại
    các khung giữa rồi nối CHUỖI cạnh liền khung (không bao giờ tạo cạnh
    nhảy t→t+k — metric bỏ hẳn cạnh nhảy).

    STITCH_COLLISION_UM > 0 sẽ chặn nếu đường nội suy đụng node đã có —
    MẶC ĐỊNH TẮT vì hành lang nối blob-break đi đúng qua node CoM của
    track bạn đồng hành; metric tự collapse cạnh trùng (merge-collapse)
    nên chuỗi song song không tạo FP cạnh, còn node thừa chỉ bị phạt nhẹ.

    Trả về (n_stitch, n_interp, n_edge); sửa `rows` tại chỗ."""
    if not STITCH_ENABLED:
        return 0, 0, 0
    node_rows = [r for r in rows if r['row_type'] == 'node']
    edge_rows = [r for r in rows if r['row_type'] == 'edge']
    has_out = {r['source_id'] for r in edge_rows}
    has_in = {r['target_id'] for r in edge_rows}

    ends_by_t, starts_by_t, nodes_by_t = {}, {}, {}
    for r in node_rows:
        if r['node_id'] not in meta:
            continue
        nodes_by_t.setdefault(r['t'], []).append(r['node_id'])
        if r['node_id'] not in has_out:
            ends_by_t.setdefault(r['t'], []).append(r['node_id'])
        if r['node_id'] not in has_in:
            starts_by_t.setdefault(r['t'], []).append(r['node_id'])

    def _pos(nid):
        return np.array(meta[nid][1:4])          # (z, y, x) µm float

    def _blocked(mid_pos, t):
        """đường nội suy đụng node đã có (chỉ khi bật collision check)."""
        if STITCH_COLLISION_UM <= 0:
            return False
        for nid in nodes_by_t.get(t, ()):         # noqa: B007
            if float(np.linalg.norm(_pos(nid) - mid_pos)) < STITCH_COLLISION_UM:
                return True
        return False

    dataset = rows[0]['dataset'] if rows else ''
    used_e, used_s = set(), set()
    added_nodes, added_edges = [], []
    n_stitch = 0

    for gap in range(1, STITCH_GAP_MAX + 1):
        gate = STITCH_GATE_UM + STITCH_GATE_PER_GAP * (gap - 1)
        for te in sorted(ends_by_t):
            E = [e for e in ends_by_t[te] if e not in used_e]
            S = [s for s in starts_by_t.get(te + gap, []) if s not in used_s]
            if not E or not S:
                continue
            cand = []
            for e in E:
                pe = _pos(e)
                me = meta[e][4]
                for s in S:
                    ps = _pos(s)
                    d = float(np.linalg.norm(ps - pe))
                    if d > gate:
                        continue    # d = 0 hợp lệ: tế bào đứng yên mờ rồi sáng lại
                    ms = meta[s][4]
                    if me > 0 and ms > 0 and abs(np.log(ms / me)) > STITCH_MAX_LOGMASS:
                        continue
                    ok = True
                    for k in range(1, gap):
                        pm = pe + (ps - pe) * (k / gap)
                        if _blocked(pm, te + k):
                            ok = False
                            break
                    if ok:
                        cand.append((d, e, s))
            if not cand:
                continue
            ei = sorted({e for _, e, _ in cand})
            si = sorted({s for _, _, s in cand})
            ie = {e: i for i, e in enumerate(ei)}
            isx = {s: i for i, s in enumerate(si)}
            D = np.full((len(ei), len(si)), BIG)
            for d, e, s in cand:
                D[ie[e], isx[s]] = d
            ri, ci = linear_sum_assignment(D)
            for r_, c_ in zip(ri, ci):
                if D[r_, c_] >= BIG:
                    continue
                e, s = ei[r_], si[c_]
                used_e.add(e)
                used_s.add(s)
                pe, ps = _pos(e), _pos(s)
                chain = e
                for k in range(1, gap):
                    pm = pe + (ps - pe) * (k / gap)          # µm
                    vf = pm / SCALE                             # voxel float
                    mid = next(NODE_ID)
                    meta[mid] = (te + k, float(pm[0]), float(pm[1]), float(pm[2]),
                                 (meta[e][4] + meta[s][4]) / 2.0)
                    nodes_by_t.setdefault(te + k, []).append(mid)
                    added_nodes.append({
                        'dataset': dataset, 'row_type': 'node', 'node_id': mid,
                        't': te + k, 'z': int(round(vf[0])), 'y': int(round(vf[1])),
                        'x': int(round(vf[2])), 'source_id': -1, 'target_id': -1,
                    })
                    added_edges.append((chain, mid))
                    chain = mid
                added_edges.append((chain, s))
                n_stitch += 1

    for a, b in added_edges:
        rows.append({
            'dataset': dataset, 'row_type': 'edge', 'node_id': -1,
            't': -1, 'z': -1, 'y': -1, 'x': -1,
            'source_id': a, 'target_id': b,
        })
    rows.extend(added_nodes)
    return n_stitch, len(added_nodes), len(added_edges)


if not os.path.isdir(DATA_DIR):
    hint = os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'không có /kaggle/input'
    raise FileNotFoundError(f'Không thấy {DATA_DIR} — hãy Add Input competition. /kaggle/input: {hint}')

test_folder_names = sorted(
    d.replace('.zarr', '') for d in os.listdir(DATA_DIR) if d.endswith('.zarr')
)
if not test_folder_names:
    raise RuntimeError(f'Không tìm thấy thư mục .zarr nào trong {DATA_DIR}: {os.listdir(DATA_DIR)[:10]}')
print(f'{len(test_folder_names)} dataset: {test_folder_names}')

all_rows = []
summary = []
STOPPED_EARLY = False

for folder_name in test_folder_names:
    zarr_path = os.path.join(DATA_DIR, folder_name + '.zarr')
    shape, dtype, chunk = read_zarr_meta(zarr_path)
    n_t = shape[0]

    TRK = Tracker()
    ds_rows = []
    node_meta = {}     # nid → (t, zµm, yµm, xµm, mass) — cho stitching hậu kiểm
    n_nodes = n_edges = n_div = 0
    t0 = time.time()

    for t in range(n_t):
        if _over_time_budget():
            print(f'!! Gần hết ngân sách {TIME_LIMIT_HOURS}h — dừng tại {folder_name} t={t}')
            STOPPED_EARLY = True
            break
        try:
            vol = load_volume(zarr_path, t, shape, dtype, chunk)
        except (FileNotFoundError, ValueError) as e:
            print(f'  [{folder_name}] lỗi đọc khung {t}: {e} — bỏ qua')
            continue
        dets = detect_nodes(vol)
        nodes, edges, divisions, interp_nodes = TRK.step(dets, t)

        for nid, d in nodes:
            ds_rows.append({
                'dataset': folder_name, 'row_type': 'node', 'node_id': nid,
                't': t, 'z': d['z'], 'y': d['y'], 'x': d['x'],
                'source_id': -1, 'target_id': -1,
            })
            node_meta[nid] = (t, d['zf'] * SCALE[0], d['yf'] * SCALE[1],
                              d['xf'] * SCALE[2], float(d['mass']))
        for nid, tk, d in interp_nodes:
            ds_rows.append({
                'dataset': folder_name, 'row_type': 'node', 'node_id': nid,
                't': tk, 'z': d['z'], 'y': d['y'], 'x': d['x'],
                'source_id': -1, 'target_id': -1,
            })
            node_meta[nid] = (tk, d['zf'] * SCALE[0], d['yf'] * SCALE[1],
                              d['xf'] * SCALE[2], float(d['mass']))
        for src, dst in edges:
            ds_rows.append({
                'dataset': folder_name, 'row_type': 'edge', 'node_id': -1,
                't': -1, 'z': -1, 'y': -1, 'x': -1,
                'source_id': src, 'target_id': dst,
            })
        n_nodes += len(nodes) + len(interp_nodes)
        n_edges += len(edges)
        n_div += len(divisions)

        if (t + 1) % 50 == 0 or t == n_t - 1:
            print(f'  [{folder_name}] {t + 1}/{n_t} khung · {n_nodes} node · '
                  f'{n_edges} cạnh · {n_div} phân bào · {time.time() - t0:.0f}s', flush=True)

    # ---- ver 4: stitching hậu kiểm — nối lại track đứt + node nội suy ----
    n_st, n_in, n_ed = stitch_tracks(ds_rows, node_meta)
    if STITCH_ENABLED and n_st:
        print(f'  [stitch {folder_name}] nối lại {n_st} track · +{n_in} node nội suy · '
              f'+{n_ed} cạnh', flush=True)
    n_nodes += n_in
    n_edges += n_ed

    all_rows.extend(ds_rows)
    if DIAGNOSE:
        print(f'  [chẩn đoán {folder_name}] {diagnose(ds_rows)}', flush=True)
    summary.append((folder_name, n_nodes, n_edges, n_div))
    print(f'== {folder_name}: {n_nodes} nodes · {n_edges} edges · {n_div} phân bào '
          f'({time.time() - t0:.0f}s)', flush=True)
    if STOPPED_EARLY:
        break

if STOPPED_EARLY:
    print('CẢNH BÁO: dừng sớm do ngân sách thời gian — kết quả chỉ một phần!')

# ---------- 5) SOÁT BẰNG MẮT (tuỳ chọn) ----------
if RUN_PREVIEW and test_folder_names:
    import matplotlib.pyplot as plt

    zarr_path = os.path.join(DATA_DIR, test_folder_names[0] + '.zarr')
    shape, dtype, chunk = read_zarr_meta(zarr_path)
    vol = load_volume(zarr_path, 0, shape, dtype, chunk)
    dets = detect_nodes(vol)

    zc = shape[1] // 2
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax in axes:
        ax.imshow(vol[zc], cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].set_title(f'{test_folder_names[0]} · t=0 · lát z={zc}')
    for d in dets:
        axes[1].plot(d['x'], d['y'], 'o', mfc='none', mec='lime', ms=9, mew=1.2)
    axes[1].set_title(f'phát hiện: {len(dets)} node (ver 4: P{PERCENTILE:g} + tách blob + stitch)')
    plt.tight_layout()
    plt.show()


In [ ]:
# ver 4 · cell 4 — XUẤT SUBMISSION
# Dán đè Cell 4 của notebook Kaggle.
# ============================================================

import pandas as pd  # tự chữa (hotfix 14/09): notebook giữ cell 1 cũ vẫn đủ pandas

COLS = ['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
submission = pd.DataFrame(all_rows, columns=COLS)
submission.index.name = 'id'
submission.to_csv('submission.csv')

n_nodes = int((submission['row_type'] == 'node').sum())
n_edges = int((submission['row_type'] == 'edge').sum())
n_div = sum(s[3] for s in summary)
print(f'Đã ghi submission.csv: {len(submission)} hàng '
      f'({n_nodes} node · {n_edges} cạnh · {n_div} phân bào)')
print(submission['row_type'].value_counts().to_string())
print(submission.head(8).to_string())


In [ ]:
# scorer · cell 2 — ENGINE: ĐỌC .GEFF + METRIC CHÍNH THỨC (port)
# ============================================================
# Port từ royerlab/kaggle-cell-tracking-competition:
#   src/tracking_cellmot/metrics.py        (edge metric + adj + summarise)
#   src/tracking_cellmot/division_metrics.py (division metric đầy đủ)
#   tracksdata DistanceMatching (ghép node per-timepoint, Hungarian ≤ 7 µm)
# Chỉ dùng numpy/scipy/pandas/blosc2 — không cần tracksdata/geff/zarr.

# ---------- 1) ĐỌC ZARR (v2 + v3) ----------
def _read_zarr_array(arr_path):
    """Đọc mảng zarr v2 (.zarray + chunk i.j.k) hoặc v3 (zarr.json + chunk c/i/j/k).
    Hỗ trợ 1D/2D, nhiều chunk, nén blosc hoặc không nén."""
    v2 = os.path.join(arr_path, '.zarray')
    if os.path.isfile(v2):
        with open(v2) as f:
            meta = json.load(f)
        dt = np.dtype(meta['dtype'])
        shape = [int(s) for s in meta['shape']]
        chunks = [int(c) for c in meta['chunks']]
        compressed = bool(meta.get('compressor'))
        prefix, sep = '', '.'
    else:
        with open(os.path.join(arr_path, 'zarr.json')) as f:
            meta = json.load(f)
        dt = np.dtype(meta['data_type'])
        shape = [int(s) for s in meta['shape']]
        chunks = [int(c) for c in
                  meta['chunk_grid']['configuration']['chunk_shape']]
        compressed = True
        prefix, sep = os.path.join('c', ''), '.'

    arr = np.zeros(shape, dtype=dt)
    grids = [range((s + c - 1) // c) for s, c in zip(shape, chunks)]
    for idx in itertools.product(*grids):
        fname = os.path.join(arr_path, prefix + sep.join(str(i) for i in idx))
        if not os.path.isfile(fname):
            continue
        with open(fname, 'rb') as f:
            raw = f.read()
        data = blosc2.decompress(raw) if compressed else raw
        block = np.frombuffer(data, dtype=dt)
        need = int(np.prod(chunks))
        block = block[:need].reshape(chunks)
        sl = tuple(
            slice(i * c, min((i + 1) * c, s))
            for i, c, s in zip(idx, chunks, shape)
        )
        inner = tuple(slice(0, s.stop - s.start) for s in sl)
        arr[sl] = block[inner]
    return arr


def read_geff(path):
    """Đọc .geff → dict:
    t, z, y, x (mảng node), node_ids, edges (E×2: source, target),
    n_total (estimated_number_of_nodes từ metadata, NaN nếu không có)."""
    v2attrs = os.path.join(path, '.zattrs')
    if os.path.isfile(v2attrs):
        with open(v2attrs) as f:
            attrs = json.load(f)
    else:
        with open(os.path.join(path, 'zarr.json')) as f:
            attrs = json.load(f).get('attributes', {})
    geff_meta = attrs.get('geff', {})

    node_ids = _read_zarr_array(os.path.join(path, 'nodes', 'ids')).astype(np.int64)
    n = len(node_ids)

    def prop(name, default):
        p = os.path.join(path, 'nodes', 'props', name, 'values')
        if not (os.path.isdir(p) or os.path.isfile(os.path.join(p, '.zarray'))
                or os.path.isfile(os.path.join(p, 'zarr.json'))):
            return np.full(n, default)
        a = _read_zarr_array(p)
        return a.astype(np.float64)

    t = prop('t', 0.0)
    z = prop('z', 0.0)
    y = prop('y', 0.0)
    x = prop('x', 0.0)

    epath = os.path.join(path, 'edges', 'ids')
    if (os.path.isfile(os.path.join(epath, '.zarray'))
            or os.path.isfile(os.path.join(epath, 'zarr.json'))):
        edges = _read_zarr_array(epath).astype(np.int64).reshape(-1, 2)
    else:
        edges = np.zeros((0, 2), dtype=np.int64)

    extra = geff_meta.get('extra', {}) or {}
    n_total = float(extra.get('estimated_number_of_nodes', np.nan))
    if not (n_total == n_total):  # NaN
        n_total = float('nan')

    return {'ids': node_ids, 't': t, 'z': z, 'y': y, 'x': x,
            'edges': edges, 'n_total': n_total}


# ---------- 2) CẤU TRÚC GRAPH NHẸ ----------
class Graph:
    __slots__ = ('pos', 't', 'edges', 'succ', 'pred', 'out_deg', 'in_deg',
                 'n_total')

    def __init__(self, g):
        self.pos = np.stack([g['z'], g['y'], g['x']], axis=1) * SCALE  # µm
        self.t = g['t'].astype(np.int64)
        # remap node_id gốc → chỉ số nội bộ 0..N-1 (giống csv_to_geffs
        # của BTC gán id mới — matching theo vị trí, không theo id)
        id2idx = {int(nid): i for i, nid in enumerate(g['ids'])}
        self.edges = []
        for s, d in g['edges']:
            s, d = int(s), int(d)
            if s not in id2idx or d not in id2idx:
                continue  # cạnh treo lơ lửng — bỏ (metric cũng bỏ)
            self.edges.append((id2idx[s], id2idx[d]))
        self.n_total = g['n_total']
        self.succ, self.pred = {}, {}
        self.out_deg, self.in_deg = {}, {}
        for s, d in self.edges:
            self.succ.setdefault(s, []).append(d)
            self.pred.setdefault(d, []).append(s)
            self.out_deg[s] = self.out_deg.get(s, 0) + 1
            self.in_deg[d] = self.in_deg.get(d, 0) + 1

    def successors(self, n):
        return self.succ.get(n, [])

    def predecessors(self, n):
        return self.pred.get(n, [])

    def dividing_nodes(self):
        return [n for n in self.succ if len(self.succ[n]) >= 2]

    def num_nodes(self):
        return len(self.t)

    def num_edges(self):
        return len(self.edges)


# ---------- 3) GHÉP NODE PER-TIMEPOINT (như tracksdata DistanceMatching) ----------
def match_nodes(pred, gt, max_distance=MAX_DISTANCE, gt_subset=None):
    """Ghép node pred ↔ gt theo từng mốc thời gian bằng Hungarian, ≤ max_distance.
    gt_subset: nếu cho, chỉ ghép với các node GT trong tập này (cho division window).
    Trả về dict pred_node → gt_node."""
    gt_idx = np.arange(gt.num_nodes())
    if gt_subset is not None:
        gt_idx = np.array(sorted(gt_subset), dtype=np.int64)
    gt_by_t = {}
    for i in gt_idx:
        gt_by_t.setdefault(int(gt.t[i]), []).append(int(i))

    matched = {}
    used_pred_by_t = {}
    for i in range(pred.num_nodes()):
        used_pred_by_t.setdefault(int(pred.t[i]), []).append(i)

    for tt, pidx in used_pred_by_t.items():
        gidx = gt_by_t.get(tt, [])
        if not gidx or not pidx:
            continue
        P = pred.pos[pidx]
        G = gt.pos[gidx]
        D = np.linalg.norm(P[:, None, :] - G[None, :, :], axis=2)
        ok = D <= max_distance
        if not ok.any():
            continue
        cost = np.where(ok, D, 1e9)
        ri, ci = linear_sum_assignment(cost)
        for r, c in zip(ri, ci):
            if D[r, c] <= max_distance:
                matched[pidx[r]] = gidx[c]  # gidx đã là chỉ số toàn cục
    return matched


# ---------- 4) EDGE METRIC (port metrics.py) ----------
def _evaluate_edge_counts(pred, gt, matched):
    """Đếm edge TP/FP/FN đúng thuật toán chính thức:
    1. bỏ cạnh không liền khung (t_dst == t_src + 1)
    2. bỏ cạnh trùgn map vào cùng cạnh GT (merge collapse) — giữ edge thấp nhất
    3. cap out-degree ≤ 2 (giữ 2 cạnh thấp nhất theo thứ tự)
    4. pred_valid: nguồn khớp GT-có-cạnh-ra HOẶC đích khớp GT-có-cạnh-vào
    5. TP = cạnh hai đầu khớp + GT có cạnh tương ứng; FP = valid − TP; FN = GT − TP
    """
    tpos = {i: int(pred.t[i]) for i in range(pred.num_nodes())}
    pred_edges = []
    for ei, (s, d) in enumerate(pred.edges):
        if s not in tpos or d not in tpos:
            continue
        if tpos[d] - tpos[s] != 1:
            continue  # metric bỏ hẳn cạnh nhảy
        pred_edges.append((ei, s, d))

    # collapse merge: cùng cặp (matched_source, matched_target)
    seen_pairs = {}
    kept = []
    for ei, s, d in pred_edges:
        ms = matched.get(s, -1)
        md = matched.get(d, -1)
        if ms >= 0 and md >= 0:
            key = (ms, md)
            if key in seen_pairs:
                continue
            seen_pairs[key] = ei
        kept.append((ei, s, d))

    # cap out-degree ≤ 2 theo thứ tự edge id
    out_count = {}
    capped = []
    for ei, s, d in kept:
        out_count[s] = out_count.get(s, 0) + 1
        if out_count[s] <= 2:
            capped.append((ei, s, d))

    gt_edge_set = set((int(a), int(b)) for a, b in gt.edges)

    tp = 0
    valid = 0
    for ei, s, d in capped:
        ms = matched.get(s, -1)
        md = matched.get(d, -1)
        src_ok = ms >= 0 and gt.out_deg.get(ms, 0) > 0
        dst_ok = md >= 0 and gt.in_deg.get(md, 0) > 0
        if src_ok or dst_ok:
            valid += 1
        if ms >= 0 and md >= 0 and (ms, md) in gt_edge_set:
            tp += 1
    fn = gt.num_edges() - tp
    fp = valid - tp
    return tp, fp, fn


# ---------- 5) DIVISION METRIC (port division_metrics.py) ----------
def _bipartite_max_matching(left, edges):
    """Maximum-cardinality bipartite matching (DFS augmenting path).
    edges: left → set(right). Trả về dict left→right."""
    match_r, match_l = {}, {}

    def augment(u, seen):
        for v in edges.get(u, ()):
            if v in seen:
                continue
            seen.add(v)
            if v not in match_r or augment(match_r[v], seen):
                match_l[u] = v
                match_r[v] = u
                return True
        return False

    for u in left:
        augment(u, set())
    return match_l


def _gt_weak_components(gt):
    comp = {}
    for seed in range(gt.num_nodes()):
        if seed in comp:
            continue
        comp[seed] = seed
        stack = [seed]
        while stack:
            cur = stack.pop()
            for nb in list(gt.successors(cur)) + list(gt.predecessors(cur)):
                if nb not in comp:
                    comp[nb] = seed
                    stack.append(nb)
    return comp


def _extract_divisions(gt):
    """div_node → {'parents', 'children', 'grandchildren', 'keep'}."""
    out = {}
    for div in gt.dividing_nodes():
        parents = list(gt.predecessors(div))
        children = list(gt.successors(div))
        grandchildren = [g for c in children for g in gt.successors(c)]
        out[div] = {'parents': parents, 'children': children,
                    'grandchildren': grandchildren,
                    'keep': set(parents) | {div} | set(children) | set(grandchildren)}
    return out


def _branch_component_evidence(pred, pred_div, child, pred_to_gt, gt_comp):
    """(component, malformed) cho một nhánh con của fork."""
    if set(pred.predecessors(child)) != {pred_div}:
        return None, True
    if child in pred_to_gt:
        return gt_comp[pred_to_gt[child]], False
    grandchildren = pred.successors(child)
    for g in grandchildren:
        if set(pred.predecessors(g)) != {child}:
            return None, True
    comps = {gt_comp[pred_to_gt[g]] for g in grandchildren if g in pred_to_gt}
    if len(comps) == 1:
        return next(iter(comps)), False
    return None, False


def _is_strongly_connected(pred, pred_div, parent_ids, daughter_ids):
    pred_parent_ids = {pred_div, *pred.predecessors(pred_div)}
    if pred_parent_ids.isdisjoint(parent_ids):
        return False
    pred_lineages = [{c, *pred.successors(c)} for c in pred.successors(pred_div)]
    edges = {
        gi: {li for li, pl in enumerate(pred_lineages) if not ids.isdisjoint(pl)}
        for gi, ids in enumerate(daughter_ids)
    }
    return len(_bipartite_max_matching(list(edges), edges)) >= 2


def evaluate_divisions(pred, gt, max_distance=MAX_DISTANCE):
    """Trả về (tp, fp, fn) cho division — port đầy đủ topology chính thức."""
    if gt.num_edges() == 0 or gt.num_nodes() == 0 or pred.num_nodes() == 0:
        return 0, 0, 0
    divs = _extract_divisions(gt)
    pred_div_nodes = set(pred.dividing_nodes())

    # fork sets từ full matching
    full_match = match_nodes(pred, gt, max_distance)
    evaluable_forks = {
        p for p in pred_div_nodes
        if p in full_match and gt.out_deg.get(full_match[p], 0) >= 1
    }
    gt_comp = _gt_weak_components(gt)
    cross_component, malformed = set(), set()
    for p in pred_div_nodes:
        branch_evidence = []
        broke = False
        for child in pred.successors(p):
            comp, bad = _branch_component_evidence(pred, p, child, full_match, gt_comp)
            if bad:
                malformed.add(p)
                broke = True
                break
            if comp is not None:
                branch_evidence.append(comp)
        if not broke and len(set(branch_evidence)) >= 2:
            cross_component.add(p)
    invalid_forks = cross_component | malformed

    candidates = {}
    considered = set()
    for div, info in divs.items():
        m = match_nodes(pred, gt, max_distance, gt_subset=info['keep'])
        node_to_gt = {p: g for p, g in m.items()}
        children = info['children']
        if len(children) < 2:
            candidates[div] = set()
            continue
        parent_side = {div, *info['parents']}
        parent_ids = {p for p, g in node_to_gt.items() if g in parent_side}
        daughter_ids = [
            {p for p, g in node_to_gt.items() if g in {c, *gt.successors(c)}}
            for c in children
        ]
        if not parent_ids or sum(bool(x) for x in daughter_ids) < 2:
            candidates[div] = set()
            continue
        local_nodes = set(parent_ids)
        for p in list(parent_ids):
            local_nodes.update(pred.successors(p))
        local_forks = local_nodes & pred_div_nodes
        considered |= local_forks
        candidates[div] = {
            f for f in local_forks - invalid_forks
            if _is_strongly_connected(pred, f, parent_ids, daughter_ids)
        }

    pairing = _bipartite_max_matching(list(candidates), candidates)
    tp = sum(1 for d in candidates if d in pairing)
    fn = len(candidates) - tp
    fp_forks = (considered | evaluable_forks | invalid_forks) - set(pairing.values())
    return tp, fn, len(fp_forks)


# ---------- 6) ĐÁNH GIÁ 1 CẶP + TỔNG HỢP ----------
def node_recall(pred, gt, matched):
    matched_gt = set(matched.values())
    return len(matched_gt) / max(1, gt.num_nodes())


def evaluate_one(pred, gt):
    if pred.num_edges() == 0 or pred.num_nodes() == 0:
        edge_tp, edge_fp = 0, 0
        edge_fn = gt.num_edges()
        matched = {}
    else:
        matched = match_nodes(pred, gt)
        edge_tp, edge_fp, edge_fn = _evaluate_edge_counts(pred, gt, matched)
    dtp, dfn, dfp = evaluate_divisions(pred, gt)
    recall = node_recall(pred, gt, matched)
    n_pred = pred.num_nodes()

    ej_den = edge_tp + edge_fp + edge_fn
    ej = edge_tp / ej_den if ej_den > 0 else float('nan')
    dj_den = dtp + dfp + dfn
    divj = dtp / dj_den if dj_den > 0 else float('nan')
    n_total = gt.n_total
    if ej == ej and n_total == n_total and n_total > 0:
        ratio = (n_pred - n_total) / n_total
        adj = max(0.0, ej * (1 - ADJUSTMENT_ALPHA * ratio))
    else:
        adj = float('nan')
    return {'edge_tp': edge_tp, 'edge_fp': edge_fp, 'edge_fn': edge_fn,
            'division_tp': dtp, 'division_fp': dfp, 'division_fn': dfn,
            'num_pred_nodes': n_pred, 'node_recall': recall,
            'n_total': n_total, 'edge_jaccard': ej, 'adj_edge_jaccard': adj,
            'division_jaccard': divj,
            'score': adj + SCORE_DIVISION_WEIGHT * divj if dj_den > 0 else adj}


def summarise(rows):
    valid = [r for r in rows if r['edge_tp'] == r['edge_tp']]
    if not valid:
        return None
    tot = {k: sum(r[k] for r in valid) for k in
           ('edge_tp', 'edge_fp', 'edge_fn', 'division_tp', 'division_fp',
            'division_fn', 'num_pred_nodes')}

    def jac(tp, fp, fn):
        d = tp + fp + fn
        return tp / d if d > 0 else float('nan')

    adj_rows = [r for r in valid if r['adj_edge_jaccard'] == r['adj_edge_jaccard']]
    if adj_rows:
        tw = sum(r['edge_tp'] + r['edge_fp'] + r['edge_fn'] for r in adj_rows)
        adj = sum((r['edge_tp'] + r['edge_fp'] + r['edge_fn']) * r['adj_edge_jaccard']
                  for r in adj_rows) / tw
    else:
        adj = float('nan')

    div_total = tot['division_tp'] + tot['division_fp'] + tot['division_fn']
    divj = jac(tot['division_tp'], tot['division_fp'], tot['division_fn']) if div_total > 0 else float('nan')
    score = adj + SCORE_DIVISION_WEIGHT * divj if div_total > 0 else adj
    return {'n': len(valid),
            'edge_jaccard': jac(tot['edge_tp'], tot['edge_fp'], tot['edge_fn']),
            'adj_edge_jaccard': adj, 'division_jaccard': divj, 'score': score,
            **tot,
            'node_recall': sum(r['node_recall'] for r in valid) / len(valid)}


# ---------- 7) NẠP SUBMISSION → DICT GRAPH THEO DATASET ----------
def load_submission_graphs(csv_path):
    df = pd.read_csv(csv_path,
                     usecols=['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x',
                              'source_id', 'target_id'])
    graphs = {}
    for name, g in df.groupby('dataset', sort=True):
        nd = g[g['row_type'] == 'node']
        ed = g[g['row_type'] == 'edge']
        geff = {
            'ids': nd['node_id'].to_numpy(np.int64),
            't': nd['t'].to_numpy(np.float64),
            'z': nd['z'].to_numpy(np.float64),
            'y': nd['y'].to_numpy(np.float64),
            'x': nd['x'].to_numpy(np.float64),
            'edges': np.stack([ed['source_id'].to_numpy(np.int64),
                               ed['target_id'].to_numpy(np.int64)], axis=1)
            if len(ed) else np.zeros((0, 2), dtype=np.int64),
            'n_total': float('nan'),
        }
        graphs[name] = Graph(geff)
    return graphs


In [ ]:
# scorer · cell 3 — CHẠY CHẤM ĐIỂM
# ============================================================

if not os.path.isdir(TRAIN_DIR):
    hint = os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'không có /kaggle/input'
    raise FileNotFoundError(f'Không thấy {TRAIN_DIR} — Add Input competition. /kaggle/input: {hint}')

if SUBMISSION_CSV and not os.path.isfile(SUBMISSION_CSV):
    raise FileNotFoundError(f'Không thấy {SUBMISSION_CSV} — chạy notebook pipeline trước rồi thử lại')

geff_names = sorted(f[:-5] for f in os.listdir(TRAIN_DIR) if f.endswith('.geff'))
if not geff_names:
    raise RuntimeError(f'Không tìm thấy .geff nào trong {TRAIN_DIR}')
if MAX_DATASETS > 0:
    geff_names = geff_names[:MAX_DATASETS]
print(f'{len(geff_names)} dataset có GT: {geff_names[:8]}{"..." if len(geff_names) > 8 else ""}')

pred_graphs = load_submission_graphs(SUBMISSION_CSV) if SUBMISSION_CSV else {}

rows = []
t0 = time.time()
for name in geff_names:
    if name not in pred_graphs:
        print(f'  SKIP {name}: không có trong submission')
        continue
    gt = Graph(read_geff(os.path.join(TRAIN_DIR, name + '.geff')))
    pred = pred_graphs[name]
    r = evaluate_one(pred, gt)
    r['dataset'] = name
    rows.append(r)
    print(f"  {name}: EJ={r['edge_jaccard']:.4f} adjEJ={r['adj_edge_jaccard']:.4f} "
          f"edge {r['edge_tp']}/{r['edge_fp']}/{r['edge_fn']} "
          f"div {r['division_tp']}/{r['division_fp']}/{r['division_fn']} "
          f"recall={r['node_recall']:.3f} n_pred={r['num_pred_nodes']}", flush=True)

s = summarise(rows)
print(f'\n=== TỔNG ({len(rows)} dataset, {time.time() - t0:.0f}s) ===')
if s:
    print(f"score                = {s['score']:.4f}")
    print(f"adj_edge_jaccard     = {s['adj_edge_jaccard']:.4f}   (edge J = {s['edge_jaccard']:.4f})")
    print(f"division_jaccard     = {s['division_jaccard']:.4f}   "
          f"(TP={s['division_tp']} FP={s['division_fp']} FN={s['division_fn']})")
    print(f"node_recall          = {s['node_recall']:.4f}")
    print(f"edge TP/FP/FN        = {s['edge_tp']}/{s['edge_fp']}/{s['edge_fn']}")
    print(f"số node dự đoán      = {s['num_pred_nodes']}")
    print()
    print('So sánh nhanh: ver 1 Kaggle = 0.198 · top 1 = 0.97')
    if s['edge_fn'] > s['edge_tp']:
        print('→ FN cạnh trội TP: ưu tiên TĂNG RECALL (hạ ngưỡng, mở gate, '
              'nối lại track đứt, nội suy).')
    if s['edge_fp'] > 0.3 * s['edge_tp']:
        print('→ FP cạnh đáng kể: kiểm tra cạnh chéo giữa 2 track gần nhau, '
              'coi chừng gán nhầm sau nội suy.')
    if s['division_fn'] > 2 * s['division_tp'] and s['division_fn'] > 0:
        print('→ bỏ sót phân bào: giảm DIV_CONFIRM_FRAMES hoặc nới gate.')
    if s['division_fp'] > s['division_tp'] and s['division_fp'] > 0:
        print('→ phân bào giả: tăng DIV_CONFIRM_FRAMES / DIV_SEP_GROWTH.')

# bảng chi tiết ra DataFrame để dễ xem
detail = pd.DataFrame(rows).set_index('dataset') if rows else None
detail
